# 🌳 Digital Forest Twin - Tree Data Importer (Python)

This notebook provides a step-by-step workflow to import tree data from CSV files into the Digital Forest Twin database.

## Workflow Overview

1. **Setup**: Load dependencies and connect to database
2. **Introspect**: Display available database schema
3. **Load CSV**: Load your data file
4. **Reference Data**: Explore species, locations, and sensor types
5. **Define Mapping**: Create column mappings (with LOOKUP support)
6. **Coordinate Mapping**: Handle lat/lon or x/y coordinates
7. **Preview**: Review data before insertion
8. **Insert**: Load data into database

## Instructions

- Run cells sequentially from top to bottom
- Modify parameters in cells as needed for your CSV file
- Use the "LOOKUP" feature during mapping to inspect CSV values
- The notebook saves your mapping as JSON for reuse

## Step 1: Setup - Load Dependencies and Connect to Database

In [ ]:
# Install/import required packages
import os
import psycopg2
from pathlib import Path
from typing import Any, Dict, List, Optional
import json

import pandas as pd
from dotenv import load_dotenv

try:
    from pyproj import Transformer

    HAS_PYPROJ = True
except ImportError:
    HAS_PYPROJ = False
    print("⚠️  pyproj not installed. Install with: pip install pyproj")

# Load environment
load_dotenv("../docker/.env")
print("✓ Dependencies loaded")

✓ Dependencies loaded


In [5]:
# Configure database connection
from pathlib import Path

# Only change directory if not already in docker folder
if not os.getcwd().endswith("docker"):
    target_dir = Path(os.getcwd())
    # Navigate up to find the project root
    while target_dir.name != "digital-twin" and target_dir.parent != target_dir:
        target_dir = target_dir.parent
    target_dir = target_dir / "docker"
    if target_dir.exists():
        os.chdir(target_dir)

print(f"Working directory: {os.getcwd()}")

POSTGRES_HOST = "localhost"
POSTGRES_USER = os.getenv("POSTGRES_USER", "postgres")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_DATABASE = os.getenv("POSTGRES_DB", "postgres")
POSTGRES_PORT = os.getenv("POSTGRES_PORT", "5432")

# Supavisor requires tenant ID in the username format: user.tenant_id
POOLER_TENANT_ID = os.getenv("POOLER_TENANT_ID", "")
if POOLER_TENANT_ID:
    POSTGRES_USER_POOLER = f"{POSTGRES_USER}.{POOLER_TENANT_ID}"
else:
    POSTGRES_USER_POOLER = POSTGRES_USER

if not POSTGRES_PASSWORD:
    print("❌ POSTGRES_PASSWORD not found in docker/.env")
else:
    print(
        f"✓ Database config: {POSTGRES_USER_POOLER}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DATABASE}"
    )

# Test connection
try:
    conn = psycopg2.connect(
        host=POSTGRES_HOST,
        user=POSTGRES_USER_POOLER,
        password=POSTGRES_PASSWORD,
        database=POSTGRES_DATABASE,
        port=POSTGRES_PORT,
    )
    conn.close()
    print("✓ Database connection successful")
except Exception as e:
    print(f"❌ Database connection failed: {e}")

Working directory: /home/maximilian_sperlich/git/digital-twin/docker
✓ Database config: postgres.digital-forest-twin-local@localhost:5432/postgres
✓ Database connection successful


## Step 2: Introspect Database Schema

In [9]:
def introspect_schema():
    """Query database schema from information_schema"""
    query = """
        SELECT
            table_schema,
            table_name,
            column_name
        FROM information_schema.columns
        WHERE table_schema IN ('shared', 'trees', 'sensor', 'pointclouds', 'environments')
        ORDER BY table_schema, table_name, ordinal_position
    """

    try:
        conn = psycopg2.connect(
            host=POSTGRES_HOST,
            user=POSTGRES_USER_POOLER,
            password=POSTGRES_PASSWORD,
            database=POSTGRES_DATABASE,
            port=POSTGRES_PORT,
        )
        df = pd.read_sql(query, conn)
        conn.close()

        # Build nested dictionary
        schema_info = {}
        for _, row in df.iterrows():
            schema = row["table_schema"]
            table = row["table_name"]
            column = row["column_name"]

            if schema not in schema_info:
                schema_info[schema] = {}
            if table not in schema_info[schema]:
                schema_info[schema][table] = []

            schema_info[schema][table].append(column)

        return schema_info
    except Exception as e:
        print(f"❌ Schema introspection failed: {e}")
        return {}


# Get schema
schema_info = introspect_schema()
print(
    f"✓ Found schema info for {sum(len(tables) for tables in schema_info.values())} tables"
)

✓ Found schema info for 41 tables


/tmp/ipykernel_186305/1365331066.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [10]:
# Display schema
print("\n" + "=" * 80)
print("DATABASE SCHEMA - Available Tables & Columns")
print("=" * 80)

for schema_name in sorted(schema_info.keys()):
    print(f"\n📦 Schema: {schema_name}")
    tables = schema_info[schema_name]
    for table_name in sorted(tables.keys()):
        columns = tables[table_name]
        print(f"  📋 {table_name} ({len(columns)} columns)")
        for i, col in enumerate(columns, 1):
            print(f"     {i:2d}. {col}")


DATABASE SCHEMA - Available Tables & Columns

📦 Schema: environments
  📋 active_environments (34 columns)
      1. variantid
      2. parentvariantid
      3. locationid
      4. scenarioid
      5. varianttypeid
      6. processid
      7. variantname
      8. startdate
      9. enddate
     10. avgtemperature_c
     11. avghumidity_percent
     12. totalprecipitation_mm
     13. avgglobalradiation
     14. avgco2_ppm
     15. avgwindspeed_ms
     16. dominantwinddirection_deg
     17. avgsoilmoisture_percent
     18. avgsoiltemperature_c
     19. soilph
     20. nutrientnitrogen_mg_kg
     21. nutrientphosphorus_mg_kg
     22. nutrientpotassium_mg_kg
     23. stressfactor
     24. description
     25. researchnotes
     26. createdat
     27. updatedat
     28. createdby
     29. updatedby
     30. duration_days
     31. is_active
     32. locationname
     33. scenarioname
     34. varianttypename
  📋 environments (29 columns)
      1. variantid
      2. parentvariantid
      3. lo

## Step 3: Load Reference Data

In [11]:
def load_reference_data():
    """Load reference tables for mapping help"""
    reference_data = {}

    try:
        conn = psycopg2.connect(
            host=POSTGRES_HOST,
            user=POSTGRES_USER_POOLER,
            password=POSTGRES_PASSWORD,
            database=POSTGRES_DATABASE,
            port=POSTGRES_PORT,
        )

        # Load Species
        reference_data["Species"] = pd.read_sql(
            "SELECT SpeciesID, CommonName, ScientificName FROM shared.Species ORDER BY CommonName",
            conn,
        )

        # Load Locations
        reference_data["Locations"] = pd.read_sql(
            "SELECT LocationID, LocationName FROM shared.Locations ORDER BY LocationName",
            conn,
        )

        # Try to load SensorTypes
        try:
            reference_data["SensorTypes"] = pd.read_sql(
                "SELECT SensorTypeID, SensorTypeName FROM sensor.SensorTypes ORDER BY SensorTypeName",
                conn,
            )
        except:
            pass

        conn.close()
        return reference_data
    except Exception as e:
        print(f"❌ Failed to load reference data: {e}")
        return {}


# Load reference data
reference_data = load_reference_data()
print(f"✓ Loaded reference data: {', '.join(reference_data.keys())}")

✓ Loaded reference data: Species, Locations, SensorTypes


/tmp/ipykernel_186305/4268738244.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  reference_data["Species"] = pd.read_sql(
/tmp/ipykernel_186305/4268738244.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  reference_data["Locations"] = pd.read_sql(
/tmp/ipykernel_186305/4268738244.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  reference_data["SensorTypes"] = pd.read_sql(


In [12]:
# Display reference data
print("\n" + "=" * 80)
print("📚 REFERENCE DATA - Use for mapping CSV values to database IDs")
print("=" * 80)

if "Species" in reference_data:
    print("\n📚 Species:")
    display(reference_data["Species"])

if "Locations" in reference_data:
    print("\n📚 Locations:")
    display(reference_data["Locations"])

if "SensorTypes" in reference_data:
    print("\n📚 SensorTypes:")
    display(reference_data["SensorTypes"])


📚 REFERENCE DATA - Use for mapping CSV values to database IDs

📚 Species:


,speciesid,commonname,scientificname
0,6,Douglas Fir,Pseudotsuga menziesii
1,1,European Beech,Fagus sylvatica
2,3,Norway Spruce,Picea abies
3,2,Pedunculate Oak,Quercus robur
4,5,Scots Pine,Pinus sylvestris
5,4,Silver Fir,Abies alba



📚 Locations:


,locationid,locationname
0,3,Black Forest Test Site
1,4,Ecosense_MixedPlot
2,1,University Forest Plot A
3,2,University Forest Plot B



📚 SensorTypes:


,sensortypeid,sensortypename
0,9,Barometric_Pressure
1,3,CO2
2,2,Humidity
3,12,Leaf_Wetness
4,4,Light
5,8,Precipitation
6,13,Sap_Flow
7,5,Soil_Moisture
8,11,Soil_Temperature
9,10,Solar_Radiation


## Step 4: Load CSV File

In [14]:
# Specify your CSV file path
CSV_PATH = "../data/mathisle_250904.csv"  # Change this to your CSV path

# Load CSV
if not Path(CSV_PATH).exists():
    print(f"❌ CSV file not found: {CSV_PATH}")
else:
    df = pd.read_csv(CSV_PATH)
    print(f"✓ Loaded CSV: {Path(CSV_PATH).name}")
    print(f"  Rows: {len(df)}")
    print(f"  Columns: {', '.join(df.columns)}")

✓ Loaded CSV: mathisle_250904.csv
  Rows: 741
  Columns: Unnamed: 0, species_short, date_time, qr_code, tree_id_fallback, gps_latitude, gps_longitude, gps_height, DBH, TreeID, species_label


In [15]:
# Display first few rows
print(f"\nFirst 3 rows of {Path(CSV_PATH).name}:")
display(df.head(3))


First 3 rows of mathisle_250904.csv:


,Unnamed: 0,species_short,date_time,qr_code,tree_id_fallback,gps_latitude,gps_longitude,gps_height,DBH,TreeID,species_label
0,1,BE,2025/03/05 13:38:01.635,https://dt.unr.uni-freiburg.de/mathisle/367,NaN,47.884879,8.088243,1046.494,0.21963,367.0,10 Beech (Fagus sylvatica )
1,2,BE,2025/03/26 16:32:10.716,https://dt.unr.uni-freiburg.de/mathisle/515,NaN,47.884718,8.088407,1046.463,0.30876,515.0,10 Beech (Fagus sylvatica )
2,3,BE,2025/03/05 13:41:49.615,https://dt.unr.uni-freiburg.de/mathisle/405,NaN,47.884768,8.087773,1052.428,0.17189,405.0,10 Beech (Fagus sylvatica )


## Step 5: Coordinate Mapping Guide

If your CSV has separate lat/lon or x/y columns, read this section to understand mapping options.

In [ ]:
print(
    """
="*80
📍 COORDINATE MAPPING - How to handle spatial data
="*80

Your CSV has coordinates in separate columns (latitude/longitude or x/y).
The database stores them as a single geometry column (Position).

MAPPING STRATEGIES:

1️⃣  COMBINED COLUMN MAPPING (Recommended)
   Format: lat_lon:EPSG_CODE or x_y:EPSG_CODE

   Examples:
   - For lat/lon columns: 'coordinates' maps to: lat_lon:EPSG:4326
   - For x/y columns: 'utm_coords' maps to: x_y:EPSG:32632

   Benefits: Automatic combination + CRS transformation

   Available column names:
   - Latitude: latitude, lat, lat_col, latitude_col, gps_latitude
   - Longitude: longitude, lon, lon_col, longitude_col, gps_longitude
   - X/Easting: x, x_col, easting, easting_col, utm_x, x_32632
   - Y/Northing: y, y_col, northing, northing_col, utm_y, y_32632

2️⃣  MANUAL GEOMETRY STRING
   Map to: shared.Locations.Position (or trees.Trees.Position, etc.)
   Create WKT format before import: POINT(lon lat)

3️⃣  SEPARATE COORDINATES (Not Recommended)
   ❌ Don't map lat/lon to separate database columns
   ✅ Combine them first using option 1 or 2

CRS SUPPORT:
   - EPSG:4326 (WGS84) - default, no transformation needed
   - EPSG:32632 (UTM Zone 32N) - automatically transforms to WGS84
   - Other EPSG codes - specify in mapping format

COMMON COORDINATES IN YOUR DATA:
   Mathisle: gps_latitude, gps_longitude (WGS84)
   EcoSense: x_32632, y_32632 (UTM Zone 32N)
"""
)

## Step 6: Define Column Mapping

In [ ]:
# Define your column mapping here
# Format: csv_column: "schema.table.column" or "lat_lon:EPSG:4326" or "x_y:EPSG:32632" or "SKIP"

mapping = {
    # Example for Mathisle trees CSV:
    # "species_short": "shared.Species.CommonName",
    # "gps_latitude": "lat_lon:EPSG:4326",  # Auto-detects gps_longitude
    # "gps_longitude": "SKIP",  # Skip since lat_lon mapping handles it
    # "DBH": "trees.Trees.FieldNotes",
    # "TreeID": "trees.Trees.FieldNotes",
    # "height": "trees.Trees.Height_m",
}

print("Mapping defined. Edit the cell above to configure for your CSV.")
print(f"\nCSV Columns: {list(df.columns)}")
print(f"\nCurrent Mapping:")
for csv_col, target in mapping.items():
    print(f"  '{csv_col}' -> {target}")

### Use LOOKUP to inspect CSV values before deciding on mapping

Edit the cell below to inspect specific columns from your CSV

In [ ]:
# Inspect a column to understand its values
column_to_inspect = "species_short"  # Change this to any column name

if column_to_inspect in df.columns:
    unique_values = df[column_to_inspect].unique()
    print(f"\nSample values from '{column_to_inspect}' ({len(unique_values)} unique):")
    for val in unique_values[:10]:
        print(f"  - {val}")
    if len(unique_values) > 10:
        print(f"  ... and {len(unique_values) - 10} more")
else:
    print(f"Column '{column_to_inspect}' not found in CSV")
    print(f"Available columns: {list(df.columns)}")

## Step 7: Save Mapping as JSON

In [ ]:
# Save mapping to JSON for reuse
mapping_path = Path(CSV_PATH).parent / f"{Path(CSV_PATH).stem}_mapping.json"

with open(mapping_path, "w") as f:
    json.dump(mapping, f, indent=2)

print(f"✓ Mapping saved to: {mapping_path}")

# Display saved mapping
with open(mapping_path, "r") as f:
    saved_mapping = json.load(f)
    print(f"\nSaved mapping:")
    print(json.dumps(saved_mapping, indent=2))

## Step 8: Process Coordinates

In [ ]:
def detect_coordinate_columns(df, format_spec):
    """Detect lat/lon or x/y columns by flexible naming"""
    col_names_lower = {col.lower(): col for col in df.columns}

    if format_spec.startswith("lat_lon"):
        # Find latitude and longitude
        lat_patterns = ["latitude", "lat", "gps_latitude", "gps_lat"]
        lon_patterns = ["longitude", "lon", "gps_longitude", "gps_lon"]

        lat_col = next(
            (col_names_lower[p] for p in lat_patterns if p in col_names_lower), None
        )
        lon_col = next(
            (col_names_lower[p] for p in lon_patterns if p in col_names_lower), None
        )

        return {"lat": lat_col, "lon": lon_col}

    elif format_spec.startswith("x_y"):
        # Find x and y
        x_patterns = ["x", "easting", "utm_x", "x_32632", "x_32633"]
        y_patterns = ["y", "northing", "utm_y", "y_32632", "y_32633"]

        x_col = next(
            (col_names_lower[p] for p in x_patterns if p in col_names_lower), None
        )
        y_col = next(
            (col_names_lower[p] for p in y_patterns if p in col_names_lower), None
        )

        return {"x": x_col, "y": y_col}

    return {}


def process_coordinates(df, mapping):
    """Process coordinate columns and create Position geometry"""
    if not HAS_PYPROJ:
        print("⚠️  pyproj not available. Coordinate processing skipped.")
        return df

    for csv_col, target in mapping.items():
        if not isinstance(target, str) or ":" not in target:
            continue

        coord_type, crs = target.split(":", 1)

        if coord_type == "lat_lon":
            coords = detect_coordinate_columns(df, target)
            if not coords["lat"] or not coords["lon"]:
                print(f"⚠️  Could not find lat/lon columns")
                continue

            lat_col = coords["lat"]
            lon_col = coords["lon"]

            print(f"📍 Processing {csv_col}: lat={lat_col}, lon={lon_col}, CRS={crs}")

            # Transform if needed
            if crs.upper() != "EPSG:4326":
                transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
                df["Position"] = df.apply(
                    lambda row: (
                        f"POINT({transformer.transform(row[lon_col], row[lat_col])[0]} {transformer.transform(row[lon_col], row[lat_col])[1]})"
                        if pd.notna(row[lat_col]) and pd.notna(row[lon_col])
                        else None
                    ),
                    axis=1,
                )
            else:
                df["Position"] = df.apply(
                    lambda row: (
                        f"POINT({row[lon_col]} {row[lat_col]})"
                        if pd.notna(row[lat_col]) and pd.notna(row[lon_col])
                        else None
                    ),
                    axis=1,
                )

            valid_count = df["Position"].notna().sum()
            print(f"✓ Created {valid_count} Position geometries")

        elif coord_type == "x_y":
            coords = detect_coordinate_columns(df, target)
            if not coords["x"] or not coords["y"]:
                print(f"⚠️  Could not find x/y columns")
                continue

            x_col = coords["x"]
            y_col = coords["y"]

            print(f"📍 Processing {csv_col}: x={x_col}, y={y_col}, CRS={crs}")

            # Transform to WGS84
            transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
            df["Position"] = df.apply(
                lambda row: (
                    f"POINT({transformer.transform(row[x_col], row[y_col])[0]} {transformer.transform(row[x_col], row[y_col])[1]})"
                    if pd.notna(row[x_col]) and pd.notna(row[y_col])
                    else None
                ),
                axis=1,
            )

            valid_count = df["Position"].notna().sum()
            print(f"✓ Transformed to WGS84")
            print(f"✓ Created {valid_count} Position geometries")

    return df


# Process coordinates if any are defined
df = process_coordinates(df, mapping)

## Step 9: Organize Data by Table

In [ ]:
def apply_mapping(df, mapping):
    """Organize data by target table"""
    table_dfs = {}

    for csv_col, target in mapping.items():
        if target is None or ":" in str(target):  # Skip special formats
            continue

        try:
            schema, table, column = target.split(".")
            table_key = f"{schema}.{table}"

            if table_key not in table_dfs:
                table_dfs[table_key] = pd.DataFrame()

            table_dfs[table_key][column] = df[csv_col]
        except ValueError:
            print(f"⚠️  Invalid mapping for '{csv_col}': {target}")

    # Add Position column if it exists
    if "Position" in df.columns:
        if "trees.Trees" in table_dfs:
            table_dfs["trees.Trees"]["Position"] = df["Position"]
        elif "sensor.Sensors" in table_dfs:
            table_dfs["sensor.Sensors"]["Position"] = df["Position"]
        elif len(table_dfs) > 0:
            first_table = list(table_dfs.keys())[0]
            table_dfs[first_table]["Position"] = df["Position"]

    return table_dfs


# Apply mapping
table_dfs = apply_mapping(df, mapping)
print(f"✓ Organized data into {len(table_dfs)} tables")

## Step 10: Preview Data

In [ ]:
# Display preview of data for each table
print("\n" + "=" * 80)
print("DATA PREVIEW - How data will be inserted into each table")
print("=" * 80)

for table_name, table_df in table_dfs.items():
    print(f"\n📊 {table_name} ({len(table_df)} rows, {len(table_df.columns)} columns)")
    print(f"   Columns: {', '.join(table_df.columns)}")
    print(f"\n   First 2 rows:")
    display(table_df.head(2))

## Step 11: Insert Data into Database (Optional)

Uncomment and run this cell to insert the prepared data into the database.

In [ ]:
# This is a template for inserting data
# Uncomment the code below and run to insert

"""
import psycopg2
from psycopg2.extras import execute_values

def insert_table_data(table_name, df):
    '''Insert DataFrame into specified table'''
    schema, table = table_name.split('.')

    try:
        conn = psycopg2.connect(
            host=POSTGRES_HOST,
            user=POSTGRES_USER_POOLER,
            password=POSTGRES_PASSWORD,
            database=POSTGRES_DATABASE,
            port=POSTGRES_PORT
        )
        cur = conn.cursor()

        # Prepare INSERT statement
        columns = ', '.join(df.columns)
        placeholders = ', '.join(['%s'] * len(df.columns))
        query = f"INSERT INTO {schema}.{table} ({columns}) VALUES ({placeholders})"

        # Convert DataFrame to list of tuples
        values = [tuple(row) for row in df.itertuples(index=False, name=None)]

        # Execute insert
        execute_values(cur, query, values)
        conn.commit()

        print(f"✓ Inserted {len(df)} rows into {table_name}")

        conn.close()
    except Exception as e:
        print(f"❌ Failed to insert into {table_name}: {e}")

# Insert all tables
for table_name, table_df in table_dfs.items():
    if len(table_df) > 0:
        insert_table_data(table_name, table_df)
"""

## Summary

✓ **What you've done:**
1. Connected to the Digital Forest Twin database
2. Introspected available database schema
3. Loaded reference data (Species, Locations, SensorTypes)
4. Loaded your CSV file
5. Defined column mappings
6. Processed coordinates (lat/lon or x/y)
7. Organized data by table
8. Previewed data before insertion

**Next steps:**
1. Review the data preview above
2. Mapping is saved to: `{mapping_path}`
3. Data is ready in the `table_dfs` variable for insertion
4. Uncomment and run Step 11 to insert into database
5. Or use the data in any other way you need

**Tips:**
- You can reuse saved mappings by loading the JSON file
- Use the LOOKUP feature (Step 6) to inspect column values
- Modify the mapping cell to change column assignments
- Run cells individually to test each step